# BT4012 — Experimental pipeline

## tl;dr
The original notebooks and submissions are preserved. This companion reviews immutable CLI runs; modeling logic lives in `pipeline.py` and `run.py`. Exact V2 reproduction must be judged from the measured differences below. No new Kaggle score is claimed.

## Context & Methods

Development: train 1–7, validate 8–21. Later historical check: train 1–21, evaluate 22–35. Both horizons are 14 steps. Validation windows do not overlap. The later period has been used before and is **not an untouched holdout**.

### Key Assumptions
V7 uses the released graph and available same-step covariates without labels. Competition permission remains unverified because local rules were absent. Unknown labels are excluded from supervised learning. Feature order has no assumed semantic interpretation.

Tables are used here for exact run receipts and metric comparisons; per-step CSVs retain the detailed evidence.

### 1. Setup

In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display
HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd() / 'experiments'
assert (HERE / 'run.py').exists()
print('Pipeline:', HERE.resolve())

Pipeline: /Users/melodygoh/bt4012_kaggle/experiments


### 2. Reproduce experiments
Set `RERUN=True` to run the fixed experiment again. Each invocation creates a new output directory. Review development output before separately running `evaluate --role historical_check`. Existing saved runs are sufficient to review the recorded results.

In [2]:
RERUN = False
if RERUN:
    for arguments in [['reproduce-v2'], ['evaluate', '--role', 'development']]:
        subprocess.run([sys.executable, str(HERE / 'run.py'), *arguments], check=True)
else:
    print('Reviewing saved completed runs. Set RERUN=True for fresh reproduction and development runs.')

Reviewing saved completed runs. Set RERUN=True for fresh reproduction and development runs.


## Data
The authoritative inputs are the four supplied CSVs in the project root. Their SHA-256 hashes and library versions are saved in each run manifest.

In [3]:
runs = sorted(p for p in (HERE / 'runs').iterdir() if (p / 'COMPLETE').exists())
audit = json.loads((HERE / 'runs/v2_reproduction/audit.json').read_text())
display(pd.Series(audit, name='Observed value').to_frame())

,Observed value
edges,234355.00000
duplicate_edges,0.00000
edges_with_both_features,175918.00000
cross_step_known_edges,0.00000
graph_nodes,203769.00000
test_without_known_predecessor_fraction,0.60317
train_rows,141772.00000
test_rows,15329.00000
labelled_rows,31235.00000
unknown_rows,110537.00000


## Results
### V2 reproduction

In [4]:
comparisons = []
for run in runs:
    if (run / 'reproduction.json').exists():
        comparisons.append({'run': run.name, **json.loads((run / 'reproduction.json').read_text())})
display(pd.DataFrame(comparisons).drop(columns=['reference_sha256']))

,run,max_absolute_difference,mean_absolute_difference,matches_at_1e_10,spearman_rank_correlation
0,v2_notebook_threads,0.044036,0.000464,False,0.999343
1,v2_reproduction,0.044036,0.000464,False,0.999343


### Frozen model comparison
Pooled ROC-AUC is the competition metric. AP is a secondary diagnostic. Do not average overlapping legacy windows as independent evidence.

In [5]:
tables = [pd.read_csv(run / 'metrics.csv').assign(run=run.name) for run in runs if (run / 'metrics.csv').exists()]
results = pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
display(results.round(6))

,fold,role,model,roc_auc,average_precision,rows,positives,run
0,development_8_21,development,original,0.993416,0.981339,9977,1680,development_initial
1,development_8_21,development,v7_graph,0.993487,0.981250,9977,1680,development_initial
2,development_8_21,development,regularized_graph,0.992687,0.979183,9977,1680,development_initial
3,historical_check_22_35,historical_check,original,0.892850,0.805404,11705,1773,historical_initial
4,historical_check_22_35,historical_check,v7_graph,0.901627,0.810814,11705,1773,historical_initial
5,historical_check_22_35,historical_check,regularized_graph,0.896788,0.804129,11705,1773,historical_initial


### Time-step variation
Macro AUC is an unweighted mean across steps with both classes; it differs from pooled AUC. The range describes historical variability, not a confidence interval.

In [6]:
step_tables = [pd.read_csv(run / 'metrics_by_step.csv').assign(run=run.name) for run in runs if (run / 'metrics_by_step.csv').exists()]
steps = pd.concat(step_tables, ignore_index=True) if step_tables else pd.DataFrame()
if not steps.empty:
    display(steps.groupby(['run','model']).roc_auc.agg(['mean','min','max','count']).round(6))

mean       min       max  count
run                 model                                                 
development_initial original           0.992381  0.976617  0.999676     14
                    regularized_graph  0.991681  0.980055  0.999135     14
                    v7_graph           0.992831  0.973124  0.999784     14
historical_initial  original           0.918830  0.723742  1.000000     14
                    regularized_graph  0.921552  0.748672  0.999941     14
                    v7_graph           0.923840  0.733944  1.000000     14

### Preservation check

In [7]:
sys.path.insert(0, str(HERE))
from pipeline import sha256
original = json.loads((HERE / 'original_checksums.json').read_text())
changed = [name for name, expected in original.items() if sha256(HERE.parent / name) != expected]
assert not changed, changed
print(f'All {len(original)} original files are byte-for-byte unchanged.')

All 25 original files are byte-for-byte unchanged.


## Takeaways
Use `AUDIT.md` for the findings and proposed next experiments and `RESULTS.md` for the measured candidate decision. No local score guarantees a private leaderboard improvement. V2 lineage remains unresolved if the comparison is not exact. Keep the preserved successful submission as the benchmark until stronger evidence justifies replacement.